# Student Performance Prediction System
**Objective:** Build a student graduation prediction system comparing Logistic Regression and Decision Tree models.

## Phase 1: Environment Setup

In [ ]:
!pip install pandas numpy scikit-learn matplotlib seaborn kaggle

## Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    confusion_matrix, accuracy_score, precision_score, 
    recall_score, f1_score, roc_curve, auc
)

# Ensure output directory exists
os.makedirs('../outputs', exist_ok=True)

## Phase 2: Data Collection & Feature Engineering
### Load Data Function

In [ ]:
def load_data(file_path):
    """
    Load the student performance dataset (student-mat.csv).
    """
    if not os.path.exists(file_path):
        print(f"File not found: {file_path}")
        return None
    
    # UCI Student Performance dataset often uses ';' as separator
    df = pd.read_csv(file_path, sep=';')
    return df

def feature_engineering(df):
    """
    Apply feature engineering as per requirements.
    """
    # 1. Total_Score: Gabungan G1, G2, dan G3
    df['Total_Score'] = df['G1'] + df['G2'] + df['G3']
    
    # 2. Fail_History: Biner apakah pernah gagal sebelumnya
    df['Fail_History'] = df['failures'].apply(lambda x: 1 if x > 0 else 0)
    
    # 3. Study_Efficiency: Rasio studytime terhadap freetime
    # Note: freetime is usually 1-5, so no division by zero risk
    df['Study_Efficiency'] = df['studytime'] / df['freetime']
    
    # 4. Final_Grade_Class: Target variable (1: Lulus jika G3 >= 10, 0: Tidak Lulus)
    df['Final_Grade_Class'] = df['G3'].apply(lambda x: 1 if x >= 10 else 0)
    
    return df

## Phase 3: Deep Preprocessing

In [ ]:
def preprocess(df):
    """
    Clean and preprocess the data.
    """
    # Check for missing values
    missing = df.isnull().sum().sum()
    print(f"Missing values count: {missing}")
    if missing > 0:
        df = df.dropna()
        
    # Define target and features
    # Drop G3 because it's directly used to create target
    # Drop G1, G2 as well to avoid data leakage if predicting 'graduation' based on start of year
    # (The prompt says use G1, G2, G3 for Total_Score, so we keep Total_Score but drop the components)
    X = df.drop(['Final_Grade_Class', 'G3'], axis=1)
    y = df['Final_Grade_Class']
    
    # Categorical features split
    binary_cols = [col for col in X.columns if X[col].nunique() == 2 and X[col].dtype == 'object']
    multi_cols = [col for col in X.columns if X[col].nunique() > 2 and X[col].dtype == 'object']
    num_cols = [col for col in X.columns if X[col].dtype in ['int64', 'float64']]
    
    # Label Encoding for binary features
    le = LabelEncoder()
    for col in binary_cols:
        X[col] = le.fit_transform(X[col])
        
    # OneHotEncoding for multi-label categorical features
    ct = ColumnTransformer(transformers=[
        ('onehot', OneHotEncoder(drop='first'), multi_cols)
    ], remainder='passthrough')
    
    X_processed = ct.fit_transform(X)
    
    # Get feature names after OHE
    ohe_features = ct.named_transformers_['onehot'].get_feature_names_out(multi_cols)
    remaining_cols = binary_cols + num_cols
    feature_names = list(ohe_features) + remaining_cols
    
    # Data Splitting (80:20 with stratify)
    X_train, X_test, y_train, y_test = train_test_split(
        X_processed, y, test_size=0.2, random_state=42, stratify=y
    )
    
    # Scaling for Logistic Regression
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    return X_train_scaled, X_test_scaled, X_train, X_test, y_train, y_test, feature_names

## Phase 4: Model Training & Hyperparameter Tuning

In [ ]:
def train_models(X_train_scaled, X_train_raw, y_train):
    """
    Train Logistic Regression and Decision Tree models.
    """
    # 1. Logistic Regression
    lr_model = LogisticRegression(max_iter=2000, solver='lbfgs', random_state=42)
    lr_model.fit(X_train_scaled, y_train)
    
    # 2. Decision Tree
    dt_model = DecisionTreeClassifier(
        max_depth=5, 
        min_samples_split=10, 
        criterion='entropy', 
        random_state=42
    )
    dt_model.fit(X_train_raw, y_train)
    
    # Cross-Validation (K-Fold, k=5)
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    lr_cv = cross_val_score(lr_model, X_train_scaled, y_train, cv=skf, scoring='accuracy')
    dt_cv = cross_val_score(dt_model, X_train_raw, y_train, cv=skf, scoring='accuracy')
    
    print(f"Logistic Regression CV Accuracy: {lr_cv.mean():.4f} (+/- {lr_cv.std():.4f})")
    print(f"Decision Tree CV Accuracy: {dt_cv.mean():.4f} (+/- {dt_cv.std():.4f})")
    
    return lr_model, dt_model

## Phase 5: Evaluation

In [ ]:
def evaluate(models, data, feature_names):
    """
    Generate performance reports and visualizations.
    """
    lr_model, dt_model = models
    X_test_scaled, X_test_raw, y_test = data
    
    # Predictions
    y_pred_lr = lr_model.predict(X_test_scaled)
    y_prob_lr = lr_model.predict_proba(X_test_scaled)[:, 1]
    
    y_pred_dt = dt_model.predict(X_test_raw)
    y_prob_dt = dt_model.predict_proba(X_test_raw)[:, 1]
    
    results = []
    
    # 1. Confusion Matrix
    fig, ax = plt.subplots(1, 2, figsize=(12, 5))
    
    sns.heatmap(confusion_matrix(y_test, y_pred_lr), annot=True, fmt='d', cmap='Blues', ax=ax[0])
    ax[0].set_title('Logistic Regression Confusion Matrix')
    ax[0].set_xlabel('Predicted')
    ax[0].set_ylabel('Actual')
    
    sns.heatmap(confusion_matrix(y_test, y_pred_dt), annot=True, fmt='d', cmap='Greens', ax=ax[1])
    ax[1].set_title('Decision Tree Confusion Matrix')
    ax[1].set_xlabel('Predicted')
    ax[1].set_ylabel('Actual')
    
    plt.tight_layout()
    plt.savefig('../outputs/confusion_matrix.png')
    plt.show()
    
    # 2. Metrics
    for name, y_pred in [('LR', y_pred_lr), ('DT', y_pred_dt)]:
        metrics = {
            'Model': name,
            'Accuracy': accuracy_score(y_test, y_pred),
            'Precision': precision_score(y_test, y_pred),
            'Recall': recall_score(y_test, y_pred),
            'F1-Score': f1_score(y_test, y_pred)
        }
        results.append(metrics)
    
    report_df = pd.DataFrame(results)
    print("\nPerformance Metrics:")
    print(report_df)
    
    # 3. ROC Curve
    fpr_lr, tpr_lr, _ = roc_curve(y_test, y_prob_lr)
    auc_lr = auc(fpr_lr, tpr_lr)
    
    fpr_dt, tpr_dt, _ = roc_curve(y_test, y_prob_dt)
    auc_dt = auc(fpr_dt, tpr_dt)
    
    plt.figure(figsize=(8, 6))
    plt.plot(fpr_lr, tpr_lr, label=f'LR (AUC = {auc_lr:.2f})')
    plt.plot(fpr_dt, tpr_dt, label=f'DT (AUC = {auc_dt:.2f})')
    plt.plot([0, 1], [0, 1], 'k--')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('ROC Curve Comparison')
    plt.legend()
    plt.savefig('../outputs/roc_curve.png')
    plt.show()
    
    # 4. Feature Importance (Top 5)
    importances = dt_model.feature_importances_
    feat_imp = pd.Series(importances, index=feature_names).sort_values(ascending=False).head(5)
    
    plt.figure(figsize=(8, 6))
    sns.barplot(x=feat_imp.values, y=feat_imp.index, palette='viridis')
    plt.title('Top 5 Most Influential Features (Decision Tree)')
    plt.xlabel('Importance')
    plt.savefig('../outputs/feature_importance.png')
    plt.show()
    
    print("\nTop 5 Features:")
    print(feat_imp)

## Main Execution
Set the file path and run the pipeline.

In [ ]:
FILE_PATH = '../data/student-mat.csv'

df = load_data(FILE_PATH)

if df is not None:
    # 1. Feature Engineering
    df = feature_engineering(df)
    
    # 2. Deep Preprocessing
    X_train_sc, X_test_sc, X_train_rw, X_test_rw, y_train, y_test, features = preprocess(df)
    
    # 3. Model Training
    lr, dt = train_models(X_train_sc, X_train_rw, y_train)
    
    # 4. Evaluation
    evaluate((lr, dt), (X_test_sc, X_test_rw, y_test), features)
else:
    print("Please place 'student-mat.csv' in the 'data' folder.")